# 03 — Create historical graph features

The small graph showed that transfers connect accounts. This notebook converts each account's earlier activity into numbers the model can learn from. We calculate sender and receiver activity using only **earlier time steps**.

We use the first 100,000 rows as a quick prototype. The final model will use a full-data workflow after these rules are understood and checked. The fraud label, system flag, account IDs, and post-transaction balances are not model inputs.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
raw_data_dir = project_root / "data" / "raw"
csv_files = sorted(raw_data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No PaySim CSV found in {raw_data_dir}")
data_path = csv_files[0]

transactions = pd.read_csv(data_path, nrows=100_000)
print(f"Loaded {len(transactions):,} rows for a feature-engineering prototype")

Loaded 100,000 rows for a feature-engineering prototype


## Calculate earlier account activity

For each account and time step, first total its transactions for that step. Then use a cumulative total from earlier steps and subtract the current step. This deliberately treats all transactions in the same step as simultaneous, so one transaction cannot use another transaction from that same step as history.

In [2]:
def add_prior_activity(rows: pd.DataFrame, account_column: str, direction: str) -> pd.DataFrame:
    """Add count and amount totals from steps strictly before the current step."""
    step_totals = (
        rows.groupby([account_column, "step"], as_index=False)
        .agg(
            step_transaction_count=("amount", "size"),
            step_amount_total=("amount", "sum"),
        )
        .sort_values([account_column, "step"])
    )
    account_groups = step_totals.groupby(account_column, sort=False)
    step_totals[f"prior_{direction}_count"] = (
        account_groups["step_transaction_count"].cumsum()
        - step_totals["step_transaction_count"]
    )
    step_totals[f"prior_{direction}_amount"] = (
        account_groups["step_amount_total"].cumsum()
        - step_totals["step_amount_total"]
    )
    history_columns = [
        account_column, "step",
        f"prior_{direction}_count", f"prior_{direction}_amount",
    ]
    return rows.merge(
        step_totals[history_columns],
        on=[account_column, "step"],
        how="left",
        validate="many_to_one",
    )

transactions = add_prior_activity(transactions, "nameOrig", "outgoing")
transactions = add_prior_activity(transactions, "nameDest", "incoming")
print("Created prior sent/received counts and amounts.")

Created prior sent/received counts and amounts.


## Choose safe model inputs

We keep the answer (`isFraud`) as the target, but never as an input. We exclude `isFlaggedFraud` because it is another system's existing alert, and exclude `newbalanceOrig` / `newbalanceDest` because they are balances after the transaction. Account IDs are used to construct history, not given directly to the model.

In [3]:
feature_columns = [
    "step", "type", "amount",
    "oldbalanceOrg", "oldbalanceDest",
    "prior_outgoing_count", "prior_outgoing_amount",
    "prior_incoming_count", "prior_incoming_amount",
]
target_column = "isFraud"
model_table = transactions[feature_columns + [target_column]].copy()

assert model_table[target_column].isin([0, 1]).all()
assert model_table[["prior_outgoing_count", "prior_incoming_count"]].ge(0).all().all()
assert not {"isFraud", "isFlaggedFraud", "newbalanceOrig", "newbalanceDest"}.intersection(feature_columns)
print(f"Feature rows: {len(model_table):,}")
print(f"Input features: {feature_columns}")
print(f"Fraud labels in this sample: {int(model_table[target_column].sum()):,}")
display(model_table.head(10))

Feature rows: 100,000
Input features: ['step', 'type', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'prior_outgoing_count', 'prior_outgoing_amount', 'prior_incoming_count', 'prior_incoming_amount']
Fraud labels in this sample: 116


,step,type,amount,oldbalanceOrg,oldbalanceDest,prior_outgoing_count,prior_outgoing_amount,prior_incoming_count,prior_incoming_amount,isFraud
0,1,PAYMENT,9839.64,170136.00,0.0,0,0.0,0,0.0,0
1,1,PAYMENT,1864.28,21249.00,0.0,0,0.0,0,0.0,0
2,1,TRANSFER,181.00,181.00,0.0,0,0.0,0,0.0,1
3,1,CASH_OUT,181.00,181.00,21182.0,0,0.0,0,0.0,1
4,1,PAYMENT,11668.14,41554.00,0.0,0,0.0,0,0.0,0
5,1,PAYMENT,7817.71,53860.00,0.0,0,0.0,0,0.0,0
6,1,PAYMENT,7107.77,183195.00,0.0,0,0.0,0,0.0,0
7,1,PAYMENT,7861.64,176087.23,0.0,0,0.0,0,0.0,0
8,1,PAYMENT,4024.36,2671.00,0.0,0,0.0,0,0.0,0
9,1,DEBIT,5337.77,41720.00,41898.0,0,0.0,0,0.0,0


## Inspect fraud-labeled examples

Compare these rows with ordinary transactions. The feature values summarize what was known about each account before that time step. The rows are still synthetic examples, and their labels are not proof about real accounts.

In [5]:
fraud_examples = model_table.loc[model_table[target_column] == 1]
display(fraud_examples.head(10))

,step,type,amount,oldbalanceOrg,oldbalanceDest,prior_outgoing_count,prior_outgoing_amount,prior_incoming_count,prior_incoming_amount,isFraud
2,1,TRANSFER,181.00,181.00,0.0,0,0.0,0,0.0,1
3,1,CASH_OUT,181.00,181.00,21182.0,0,0.0,0,0.0,1
251,1,TRANSFER,2806.00,2806.00,0.0,0,0.0,0,0.0,1
252,1,CASH_OUT,2806.00,2806.00,26202.0,0,0.0,0,0.0,1
680,1,TRANSFER,20128.00,20128.00,0.0,0,0.0,0,0.0,1
681,1,CASH_OUT,20128.00,20128.00,6268.0,0,0.0,0,0.0,1
724,1,CASH_OUT,416001.33,0.00,102.0,0,0.0,0,0.0,1
969,1,TRANSFER,1277212.77,1277212.77,0.0,0,0.0,0,0.0,1
970,1,CASH_OUT,1277212.77,1277212.77,0.0,0,0.0,0,0.0,1
1115,1,TRANSFER,35063.63,35063.63,0.0,0,0.0,0,0.0,1


## What these features mean

- `prior_outgoing_count`: how many earlier transactions this sender had made before the current step.
- `prior_outgoing_amount`: the total amount this sender had sent before the current step.
- `prior_incoming_count` and `prior_incoming_amount`: the matching history for the receiver.
- `oldbalanceOrg` and `oldbalanceDest`: balances before the current transfer.

This is a prototype on the first 100,000 rows. Next we will build the chronological train/validation/test split and then scale the feature workflow to all rows.